# خطة التطبيق العملي لخوارزمية SAA

هنحتاج نجهز بيئة العمل، ننفذ الخوارزمية، ونقيم أدائها.

## 📦 أولاً: تجهيز البيئة والنماذج

In [ ]:
# تثبيت المكتبات المطلوبة
# !pip install torch transformers accelerate sentencepiece

import torch
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer
import copy

# تحميل النموذج الأساسي ونسختين محسنتين
base_model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
# هنستخدم نسخة محسنة على الرياضيات ونسخة محسنة على الكود
math_model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # استبدلها بنسخة محسنة حقيقية
code_model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # استبدلها بنسخة محسنة حقيقية

# تحميل النماذج
base_model = AutoModelForCausalLM.from_pretrained(base_model_name, torch_dtype=torch.float32)
math_model = AutoModelForCausalLM.from_pretrained(math_model_name, torch_dtype=torch.float32)
code_model = AutoModelForCausalLM.from_pretrained(code_model_name, torch_dtype=torch.float32)

# تثبيت الأوزان للقراءة فقط (لتوفير الذاكرة)
for model in [base_model, math_model, code_model]:
    for param in model.parameters():
        param.requires_grad = False

## ⚙️ ثانياً: تنفيذ خوارزمية SAA

In [ ]:
def svd_whiten_and_align(base_model, finetuned_models, top_k_ratio=0.9):
    """
    تنفيذ خوارزمية Spectral Alignment and Averaging (SAA)
    
    Args:
        base_model: النموذج الأساسي
        finetuned_models: قائمة بالنماذج المحسنة
        top_k_ratio: نسبة التباين المطلوب الاحتفاظ بها (افتراضي 90%)
    
    Returns:
        merged_model: النموذج المدمج
    """
    print("1. حساب متجهات المهام...")
    # جمع متجهات المهام لكل النماذج
    task_vectors = []
    for ft_model in finetuned_models:
        task_vector = {}
        for (name, base_param), (_, ft_param) in zip(
            base_model.named_parameters(), ft_model.named_parameters()
        ):
            if base_param.shape == ft_param.shape:
                task_vector[name] = ft_param - base_param
        task_vectors.append(task_vector)
    
    # دمج متجهات المهام لكل طبقة
    merged_task_vector = {}
    
    for name, base_param in base_model.named_parameters():
        print(f"معالجة: {name}, الشكل: {base_param.shape}")
        
        # تجميع متجهات المهام لهذه الطبقة
        layer_vectors = []
        for tv in task_vectors:
            if name in tv:
                layer_vectors.append(tv[name])
        
        if len(layer_vectors) == 0:
            merged_task_vector[name] = torch.zeros_like(base_param)
            continue
        
        # التعامل مع مصفوفات الوزن (ثنائية الأبعاد) فقط لـ SVD
        if base_param.dim() == 2 and base_param.shape[0] > 1 and base_param.shape[1] > 1:
            # الخطوة 2: تحليل SVD للوزن الأساسي
            U, S, Vh = torch.linalg.svd(base_param.float(), full_matrices=False)
            V = Vh.T
            
            # تبييض متجهات المهام
            whitened_vectors = []
            for lv in layer_vectors:
                # تحويل لمجال SVD: Z = Σ⁺ U^T ΔW V
                S_inv = torch.where(S > 1e-6, 1.0/S, torch.zeros_like(S))
                Z = torch.diag(S_inv) @ U.T @ lv.float() @ V
                whitened_vectors.append(Z.flatten())
            
            # تجميع المتجهات المبيضة في مصفوفة
            Z_matrix = torch.stack(whitened_vectors, dim=0)  # (n_models, dim)
            
            # الخطوة 3: تحليل PCA في الفضاء المبيض
            Z_mean = Z_matrix.mean(dim=0, keepdim=True)
            Z_centered = Z_matrix - Z_mean
            
            # حساب PCA (باستخدام SVD للمصفوفة المركزية)
            U_pca, S_pca, Vh_pca = torch.linalg.svd(Z_centered.T, full_matrices=False)
            
            # اختيار عدد المكونات الرئيسية التي تفسر top_k_ratio من التباين
            if S_pca.numel() > 0:
                explained_var_ratio = S_pca**2 / (S_pca**2).sum()
                cumsum = torch.cumsum(explained_var_ratio, dim=0)
                k = torch.searchsorted(cumsum, top_k_ratio) + 1
                k = min(k, len(S_pca))
            else:
                k = 0
            
            # الخطوة 4: محاذاة الإشارة
            aligned_vectors = []
            for i in range(Z_matrix.shape[0]):
                z_centered = Z_centered[i:i+1]  # (1, dim)
                proj = z_centered @ U_pca  # (1, n_components)
                
                aligned_proj = torch.zeros_like(proj)
                for j in range(k):
                    # تحديد إشارة الإجماع للمكون j
                    signs = torch.sign(proj[:, j])  # كل النماذج
                    majority_sign = torch.sign(signs.sum())
                    if majority_sign == 0:
                        majority_sign = 1.0
                    # محاذاة: استخدام القيمة المطلقة مع إشارة الإجماع
                    aligned_proj[:, j] = majority_sign * torch.abs(proj[:, j])
                
                # إعادة بناء المتجه المحاذي
                aligned_z = Z_mean.squeeze() + aligned_proj @ U_pca.T
                aligned_vectors.append(aligned_z)
            
            # الخطوة 5: متوسط المتجهات المحاذية
            z_merged = torch.stack(aligned_vectors).mean(dim=0)
            
            # الخطوة 6: عكس التبييض والعودة لفضاء المعاملات الأصلي
            Z_merged = z_merged.reshape(base_param.shape)
            merged_update = U @ torch.diag(S) @ Z_merged @ V.T
            merged_task_vector[name] = merged_update.to(base_param.dtype)
        else:
            # للمتجهات (التحيزات، LayerNorm): متوسط مباشر
            stacked = torch.stack([lv.float() for lv in layer_vectors], dim=0)
            merged_task_vector[name] = stacked.mean(dim=0).to(base_param.dtype)
    
    print("2. بناء النموذج المدمج...")
    # بناء النموذج النهائي
    merged_model = copy.deepcopy(base_model)
    for name, param in merged_model.named_parameters():
        if name in merged_task_vector:
            param.data = param.data + merged_task_vector[name].to(param.device)
    
    return merged_model

# تجربة الخوارزمية
print("بدء عملية الدمج باستخدام SAA...")
merged_model_saa = svd_whiten_and_align(
    base_model, 
    [math_model, code_model],
    top_k_ratio=0.9
)
print("تم الدمج بنجاح!")

## 📊 ثالثاً: تقييم أداء النموذج المدمج

للتقييم، هنستخدم معيار HellaSwag أو Arc-Easy:

In [ ]:
from lm_eval import evaluator
from lm_eval.models.huggingface import HFLM

# تحويل النماذج لصيغة تقييم lm_eval
base_lm = HFLM(pretrained=base_model, tokenizer=tokenizer)
merged_lm_saa = HFLM(pretrained=merged_model_saa, tokenizer=tokenizer)

# تقييم على HellaSwag
print("\nتقييم النموذج الأساسي:")
results_base = evaluator.simple_evaluate(
    model=base_lm,
    tasks=["hellaswag"],
    num_fewshot=0,
    limit=100
)

print("تقييم النموذج المدمج (SAA):")
results_saa = evaluator.simple_evaluate(
    model=merged_lm_saa,
    tasks=["hellaswag"],
    num_fewshot=0,
    limit=100
)

print(f"\nالدقة - النموذج الأساسي: {results_base['results']['hellaswag']['acc,none']:.4f}")
print(f"الدقة - النموذج المدمج (SAA): {results_saa['results']['hellaswag']['acc,none']:.4f}")

## 📈 رابعاً: مقارنة مع الدمج الخطي (LERP)

In [ ]:
def lerp_merge(base_model, finetuned_models):
    """دمج خطي بسيط للمقارنة"""
    merged = copy.deepcopy(base_model)
    alpha = 1.0 / len(finetuned_models)
    
    for name, param in merged.named_parameters():
        ft_sum = None
        for ft in finetuned_models:
            ft_param = dict(ft.named_parameters())[name]
            if ft_sum is None:
                ft_sum = (ft_param - dict(base_model.named_parameters())[name]) * alpha
            else:
                ft_sum += (ft_param - dict(base_model.named_parameters())[name]) * alpha
        param.data = param.data + ft_sum.to(param.device)
    
    return merged

merged_model_lerp = lerp_merge(base_model, [math_model, code_model])
merged_lm_lerp = HFLM(pretrained=merged_model_lerp, tokenizer=tokenizer)

results_lerp = evaluator.simple_evaluate(
    model=merged_lm_lerp,
    tasks=["hellaswag"],
    num_fewshot=0,
    limit=100
)

print(f"الدقة - النموذج المدمج (LERP): {results_lerp['results']['hellaswag']['acc,none']:.4f}")

## 🎯 خامساً: تحليل النتائج المتوقع

بعد تشغيل الكود، هيكون عندنا ثلاثة أرقام:

    الدقة الأساسية (Base): خط أساس للنموذج قبل أي دمج

    دقة SAA: أداء خوارزميتنا الجديدة

    دقة LERP: أداء الدمج التقليدي

النتائج المحتملة:

    تفوق SAA: دليل قوي على أصالة وفعالية الخوارزمية

    تساوي الأداء: الخوارزمية صحيحة لكنها لا تقدم تحسينًا

    تراجع SAA: الخوارزمية بها ثغرات تحتاج لإصلاح

## 📝 ملاحظات مهمة للتطبيق

    اختيار النماذج: استبدل النماذج الوهمية بنسخ حقيقية محسنة على مهام مختلفة

    الذاكرة: لو واجهت مشاكل في VRAM، استخدم TinyLlama-1.1B أو Pythia-160M

    وقت التنفيذ: توقع أن تستغرق عملية الـ SVD الأولى بعض الوقت

جاهزين لنشوف نتائج أول اختراع حقيقي لمنهجيتنا!